In [1]:
import requests
import pandas as pd

import io
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone

In [2]:
BASE_URL = "http://127.0.0.1:25503/v3"


def get_json(endpoint: str, params: dict | None = None):
    params = params or {}
    params["format"] = "json"

    r = requests.get(f"{BASE_URL}{endpoint}", params=params, timeout=30)
    r.raise_for_status()
    return r.json()


def check_symbol_exists(symbol: str = "CVX") -> bool:
    symbols = get_json("/option/list/symbols")
    return symbol in symbols


def get_expirations(symbol: str = "CVX") -> pd.DataFrame:
    data = get_json("/option/list/expirations", {"symbol": symbol})
    df = pd.DataFrame(data)
    if not df.empty:
        df["expiration"] = pd.to_datetime(df["expiration"])
        df = df.sort_values("expiration").reset_index(drop=True)
    return df


def get_option_quotes(symbol: str = "CVX", expiration: str = "*") -> pd.DataFrame:
    """
    expiration puede ser:
    - '*' para todas las expiraciones en quote
    - 'YYYY-MM-DD' para una expiración específica
    """
    data = get_json(
        "/option/snapshot/quote",
        {
            "symbol": symbol,
            "expiration": expiration,
        },
    )

    df = pd.DataFrame(data)
    if df.empty:
        return df

    numeric_cols = ["strike", "bid", "ask", "bid_size", "ask_size"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if {"bid", "ask"}.issubset(df.columns):
        df["mid"] = (df["bid"] + df["ask"]) / 2

    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    cols_order = [
        "timestamp", "symbol", "expiration", "right", "strike",
        "bid_size", "bid", "ask", "ask_size", "mid"
    ]
    cols_order = [c for c in cols_order if c in df.columns] + \
                 [c for c in df.columns if c not in cols_order]

    return df[cols_order].sort_values(["expiration", "right", "strike"]).reset_index(drop=True)


def get_option_trades(symbol: str = "CVX", expiration: str = None) -> pd.DataFrame:
    """
    trade snapshot requiere expiration.
    """
    if not expiration:
        raise ValueError("Debes indicar una expiración para snapshot/trade, por ejemplo '2026-05-15'.")

    data = get_json(
        "/option/snapshot/trade",
        {
            "symbol": symbol,
            "expiration": expiration,
        },
    )

    df = pd.DataFrame(data)
    if df.empty:
        return df

    numeric_cols = ["strike", "price", "size", "exchange", "condition"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    cols_order = [
        "timestamp", "symbol", "expiration", "right", "strike",
        "price", "size", "exchange", "condition"
    ]
    cols_order = [c for c in cols_order if c in df.columns] + \
                 [c for c in df.columns if c not in cols_order]

    return df[cols_order].sort_values(["right", "strike"]).reset_index(drop=True)

In [3]:
def td_get_csv(endpoint: str, params: dict, timeout: int = 120) -> pd.DataFrame:
    params = dict(params)
    params["format"] = "csv"

    r = requests.get(f"{BASE_URL}{endpoint}", params=params, timeout=timeout)
    r.raise_for_status()

    text = r.text.strip()
    if not text:
        return pd.DataFrame()

    return pd.read_csv(io.StringIO(text))


def chunk_dates(start_date: str, end_date: str, chunk_days: int = 30):
    start = datetime.strptime(start_date, "%Y%m%d").date()
    end = datetime.strptime(end_date, "%Y%m%d").date()

    current = start
    while current <= end:
        chunk_end = min(current + timedelta(days=chunk_days - 1), end)
        yield current.strftime("%Y%m%d"), chunk_end.strftime("%Y%m%d")
        current = chunk_end + timedelta(days=1)


def safe_end_date(end_date: str) -> str:
    requested_end = datetime.strptime(end_date, "%Y%m%d").date()
    yesterday = datetime.now().date() - timedelta(days=1)
    safe_end = min(requested_end, yesterday)
    return safe_end.strftime("%Y%m%d")


def download_option_eod_chunked(
    ticker: str,
    start_date: str,
    end_date: str,
    strike_range: int | None = 8,
    max_dte: int | None = 120,
) -> pd.DataFrame:
    end_date = safe_end_date(end_date)

    parts = []
    for s, e in chunk_dates(start_date, end_date, chunk_days=30):
        params = {
            "symbol": ticker,
            "expiration": "*",
            "start_date": s,
            "end_date": e,
            "right": "both",
            "strike": "*",
        }
        if strike_range is not None:
            params["strike_range"] = strike_range
        if max_dte is not None:
            params["max_dte"] = max_dte

        tmp = td_get_csv("/option/history/eod", params)
        if not tmp.empty:
            parts.append(tmp)

    if not parts:
        raise ValueError("No se descargaron datos de option/history/eod.")

    return pd.concat(parts, ignore_index=True)


def download_stock_eod_chunked(
    ticker: str,
    start_date: str,
    end_date: str,
) -> pd.DataFrame:
    end_date = safe_end_date(end_date)

    parts = []
    for s, e in chunk_dates(start_date, end_date, chunk_days=30):
        params = {
            "symbol": ticker,
            "start_date": s,
            "end_date": e,
        }

        tmp = td_get_csv("/stock/history/eod", params)
        if not tmp.empty:
            parts.append(tmp)

    if not parts:
        raise ValueError("No se descargaron datos de stock/history/eod.")

    return pd.concat(parts, ignore_index=True)


def build_base_options_dataset(
    ticker: str,
    start_date: str,
    end_date: str,
    strike_range: int | None = 8,
    max_dte: int | None = 120,
) -> pd.DataFrame:
    opt = download_option_eod_chunked(
        ticker=ticker,
        start_date=start_date,
        end_date=end_date,
        strike_range=strike_range,
        max_dte=max_dte,
    )

    stk = download_stock_eod_chunked(
        ticker=ticker,
        start_date=start_date,
        end_date=end_date,
    )

    opt["created"] = pd.to_datetime(opt["created"], errors="coerce")
    opt["last_trade"] = pd.to_datetime(opt["last_trade"], errors="coerce")
    opt["expiration"] = pd.to_datetime(opt["expiration"], errors="coerce")
    opt["date"] = opt["created"].dt.date

    stk["created"] = pd.to_datetime(stk["created"], errors="coerce")
    stk["date"] = stk["created"].dt.date

    stk_small = stk[["date", "close"]].rename(columns={"close": "underlying_price"})
    df = opt.merge(stk_small, on="date", how="left")

    df["ticker"] = df["symbol"]
    df["snapshot_utc"] = pd.to_datetime(df["created"], utc=True, errors="coerce")
    df["expiration_date"] = df["expiration"].dt.date
    df["option_type"] = df["right"].astype(str).str.lower()
    df["contract_symbol"] = (
        df["ticker"].astype(str)
        + "_"
        + pd.to_datetime(df["expiration_date"]).dt.strftime("%Y%m%d")
        + "_"
        + df["option_type"].map({"call": "C", "put": "P"}).fillna("?")
        + "_"
        + df["strike"].map(lambda x: f"{float(x):.3f}")
    )
    df["last_trade_date_utc"] = df["last_trade"]
    df["last_price"] = df["close"]
    df["mid"] = (df["bid"] + df["ask"]) / 2

    key_cols = ["ticker", "expiration_date", "strike", "option_type"]
    df = df.sort_values(key_cols + ["date"]).reset_index(drop=True)
    df["prev_last_price"] = df.groupby(key_cols)["last_price"].shift(1)
    df["change"] = df["last_price"] - df["prev_last_price"]
    df["percent_change"] = (df["change"] / df["prev_last_price"]) * 100.0

    df["in_the_money"] = pd.NA
    call_mask = df["option_type"] == "call"
    put_mask = df["option_type"] == "put"

    df.loc[call_mask, "in_the_money"] = (
        df.loc[call_mask, "underlying_price"] > df.loc[call_mask, "strike"]
    )
    df.loc[put_mask, "in_the_money"] = (
        df.loc[put_mask, "underlying_price"] < df.loc[put_mask, "strike"]
    )

    df["open_interest"] = pd.NA
    df["implied_volatility"] = pd.NA
    df["contract_size"] = pd.NA
    df["currency"] = "USD"
    df["source"] = "ThetaData"
    df["downloaded_at_utc"] = datetime.now(timezone.utc)

    final_cols = [
        "ticker",
        "snapshot_utc",
        "expiration_date",
        "option_type",
        "contract_symbol",
        "last_trade_date_utc",
        "strike",
        "last_price",
        "bid",
        "ask",
        "mid",
        "change",
        "percent_change",
        "volume",
        "open_interest",
        "implied_volatility",
        "in_the_money",
        "contract_size",
        "currency",
        "underlying_price",
        "source",
        "downloaded_at_utc",
        "date",
    ]

    return df[final_cols].copy()


In [20]:
df = build_base_options_dataset(
    ticker="AMZN",
    start_date="20260128",
    end_date="20260428",   # el código lo recorta automáticamente a ayer
    strike_range=30,
    max_dte=120,
)

df.to_csv("AMZN_options_base_3m.csv", index=False)

In [21]:
df_opt_cv = pd.read_csv("./AMZN_options_base_3m.csv")

In [22]:
df_opt_cv.tail()

,ticker,snapshot_utc,expiration_date,option_type,contract_symbol,last_trade_date_utc,strike,last_price,bid,ask,...,volume,open_interest,implied_volatility,in_the_money,contract_size,currency,underlying_price,source,downloaded_at_utc,date
74527,AMZN,2026-04-24 17:23:47.318000+00:00,2026-08-21,call,AMZN_20260821_C_380.000,2026-04-24 15:59:37.594,380.0,1.33,1.30,1.35,...,140,NaN,NaN,False,NaN,USD,263.99,ThetaData,2026-04-28 23:43:00.682899+00:00,2026-04-24
74528,AMZN,2026-04-27 17:22:05.280000+00:00,2026-08-21,call,AMZN_20260821_C_380.000,2026-04-27 15:35:22.868,380.0,1.27,1.18,1.28,...,415,NaN,NaN,False,NaN,USD,261.12,ThetaData,2026-04-28 23:43:00.682899+00:00,2026-04-27
74529,AMZN,2026-04-23 17:15:22.454000+00:00,2026-08-21,put,AMZN_20260821_P_380.000,2026-04-23 00:00:00.000,380.0,0.00,123.35,126.70,...,0,NaN,NaN,True,NaN,USD,255.08,ThetaData,2026-04-28 23:43:00.682899+00:00,2026-04-23
74530,AMZN,2026-04-24 17:23:47.318000+00:00,2026-08-21,put,AMZN_20260821_P_380.000,2026-04-24 00:00:00.000,380.0,0.00,114.20,117.70,...,0,NaN,NaN,True,NaN,USD,263.99,ThetaData,2026-04-28 23:43:00.682899+00:00,2026-04-24
74531,AMZN,2026-04-27 17:22:05.280000+00:00,2026-08-21,put,AMZN_20260821_P_380.000,2026-04-27 00:00:00.000,380.0,0.00,117.00,120.55,...,0,NaN,NaN,True,NaN,USD,261.12,ThetaData,2026-04-28 23:43:00.682899+00:00,2026-04-27


In [10]:
for i in df_opt_cv.columns:
    print(i)

ticker
snapshot_utc
expiration_date
option_type
contract_symbol
last_trade_date_utc
strike
last_price
bid
ask
mid
change
percent_change
volume
open_interest
implied_volatility
in_the_money
contract_size
currency
underlying_price
source
downloaded_at_utc
date


In [32]:
df_opt_cv["date"] = pd.to_datetime(df_opt_cv["date"])
df_opt_cv["expiration_date"] = pd.to_datetime(df_opt_cv["expiration_date"])
df_opt_cv["option_type"] = df_opt_cv["option_type"].astype(str).str.lower()

df_opt_cv["mid"] = (df_opt_cv["bid"] + df_opt_cv["ask"]) / 2
df_opt_cv["days_to_expiration"] = (
    df_opt_cv["expiration_date"] - df_opt_cv["date"]
).dt.days
df_opt_cv["T"] = df_opt_cv["days_to_expiration"] / 365.0
df_opt_cv["moneyness"] = df_opt_cv["underlying_price"] / df_opt_cv["strike"]

df_opt_cv["is_valid_quote"] = (
    (df_opt_cv["bid"] > 0)
    & (df_opt_cv["ask"] > 0)
    & (df_opt_cv["ask"] >= df_opt_cv["bid"])
    & (df_opt_cv["mid"] > 0)
    & (df_opt_cv["strike"] > 0)
    & (df_opt_cv["underlying_price"] > 0)
    & (df_opt_cv["T"] > 0)
)

df_opt_cv["is_otm"] = (
    ((df_opt_cv["option_type"] == "call") & (df_opt_cv["strike"] >= df_opt_cv["underlying_price"]))
    | ((df_opt_cv["option_type"] == "put") & (df_opt_cv["strike"] <= df_opt_cv["underlying_price"]))
)

surface_candidates = (
    df_opt_cv.loc[df_opt_cv["is_valid_quote"]]
    .groupby("date")
    .agg(
        valid_quotes=("contract_symbol", "size"),
        expirations=("expiration_date", "nunique"),
        strikes=("strike", "nunique"),
        otm_quotes=("is_otm", "sum"),
    )
    .sort_values(["valid_quotes", "expirations", "strikes"], ascending=False)
)

surface_candidates.head(10)

,valid_quotes,expirations,strikes,otm_quotes
date,,,,
2026-04-14,425,14,24,209
2026-04-09,416,14,24,207
2026-03-26,409,14,24,200
2026-01-29,403,14,24,201
2026-04-13,399,13,24,199
2026-04-16,394,13,24,194
2026-04-15,393,13,24,194
2026-03-31,388,14,24,193
2026-03-27,387,13,24,193


In [33]:
surface_date = surface_candidates.index[0]

df_surface_ready = (
    df_opt_cv.loc[
        (df_opt_cv["date"] == surface_date)
        & df_opt_cv["is_valid_quote"]
        & df_opt_cv["is_otm"]
    ]
    .sort_values(["expiration_date", "option_type", "strike"])
    .reset_index(drop=True)
)

df_surface_ready = df_surface_ready.loc[:, [
    "ticker",
    "date",
    "expiration_date",
    "option_type",
    "strike",
    "underlying_price",
    "bid",
    "ask",
    "mid",
    "days_to_expiration",
    "T",
    "moneyness",
    "contract_symbol",
    "volume",
]]

print(f"Fecha candidata para superficie: {surface_date.date()}")
print(f"Nodos OTM validos: {len(df_surface_ready)}")
print(f"Vencimientos: {df_surface_ready['expiration_date'].nunique()}")
print(f"Strikes: {df_surface_ready['strike'].nunique()}")

df_surface_ready.head(10)

Fecha candidata para superficie: 2026-04-14
Nodos OTM validos: 209
Vencimientos: 14
Strikes: 24


,ticker,date,expiration_date,option_type,strike,underlying_price,bid,ask,mid,days_to_expiration,T,moneyness,contract_symbol,volume
0,AMZN,2026-04-14,2026-04-15,call,250.0,249.02,1.27,1.31,1.290,1,0.00274,0.996080,AMZN_20260415_?_250.000,62767
1,AMZN,2026-04-14,2026-04-15,call,252.5,249.02,0.56,0.58,0.570,1,0.00274,0.986218,AMZN_20260415_?_252.500,33750
2,AMZN,2026-04-14,2026-04-15,call,255.0,249.02,0.25,0.27,0.260,1,0.00274,0.976549,AMZN_20260415_?_255.000,17615
3,AMZN,2026-04-14,2026-04-15,call,257.5,249.02,0.14,0.15,0.145,1,0.00274,0.967068,AMZN_20260415_?_257.500,6385
4,AMZN,2026-04-14,2026-04-15,call,260.0,249.02,0.08,0.09,0.085,1,0.00274,0.957769,AMZN_20260415_?_260.000,7007
5,AMZN,2026-04-14,2026-04-15,put,225.0,249.02,0.01,0.03,0.020,1,0.00274,1.106756,AMZN_20260415_?_225.000,618
6,AMZN,2026-04-14,2026-04-15,put,227.5,249.02,0.01,0.05,0.030,1,0.00274,1.094593,AMZN_20260415_?_227.500,793
7,AMZN,2026-04-14,2026-04-15,put,230.0,249.02,0.02,0.04,0.030,1,0.00274,1.082696,AMZN_20260415_?_230.000,6283
8,AMZN,2026-04-14,2026-04-15,put,232.5,249.02,0.04,0.06,0.050,1,0.00274,1.071054,AMZN_20260415_?_232.500,2134
9,AMZN,2026-04-14,2026-04-15,put,235.0,249.02,0.05,0.08,0.065,1,0.00274,1.059660,AMZN_20260415_?_235.000,4232


In [35]:
df_surface_ready.to_csv("AMZN_surface_data.csv")

In [34]:
surface_summary = (
    df_surface_ready.groupby("expiration_date")
    .agg(
        nodes=("contract_symbol", "size"),
        strike_min=("strike", "min"),
        strike_max=("strike", "max"),
        moneyness_min=("moneyness", "min"),
        moneyness_max=("moneyness", "max"),
    )
)

surface_summary

,nodes,strike_min,strike_max,moneyness_min,moneyness_max
expiration_date,,,,,
2026-04-15,15,225.0,260.0,0.957769,1.106756
2026-04-17,16,222.5,260.0,0.957769,1.119191
2026-04-20,16,222.5,260.0,0.957769,1.119191
2026-04-22,16,222.5,260.0,0.957769,1.119191
2026-04-24,16,222.5,260.0,0.957769,1.119191
2026-04-27,8,225.0,260.0,0.957769,1.106756
2026-04-29,10,230.0,280.0,0.889357,1.082696
2026-05-01,16,222.5,260.0,0.957769,1.119191
2026-05-08,16,205.0,280.0,0.889357,1.214732
